In [22]:
import sys
import os
import numpy as np
import torch
import tensorflow
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Get the absolute path of the folder containing your notebook
current_path = os.path.abspath(os.getcwd())

# Force Python to look here first
if current_path not in sys.path:
    sys.path.insert(0, current_path)

print(f"Python is now looking in: {current_path}")

from options import Options

# 1. TimeGAN model
from lib.timegan import TimeGAN
# 2. Data loading
from lib.data import real_data_loading, sine_data_generation

Python is now looking in: /home/tunabeluga/ECE479/TimeGAN-pytorch


### Parameters

For stock data, the feature dimension is fixed by the CSV columns. For sine data, the feature dimension is manually chosen.

#### Stock data

- `data_name = "stock"`
- `seq_len = 24` — each sample contains 24 trading days
- `z_dim = 6` — matches the 6 stock features
- `hidden_dim = 24` — internal model width
- `num_layer = 2` — number of recurrent layers
- `batch_size = 128`
- `iteration = 10` for debugging, `5000+` for actual training
- `metric_iteration = 1` for debugging, `5–10` for final evaluation

#### Sine data

- `data_name = "sine"`
- `no = 10000` — number of generated sequences
- `seq_len = 24` — each sample contains 24 synthetic time steps
- `dim = 5` — number of sine-wave features
- `z_dim = 5` — matches `dim`
- `hidden_dim = 24`
- `num_layer = 2`
- `batch_size = 128`
- `iteration = 10` for debugging, `5000+` for actual training
- `metric_iteration = 1` for debugging, `5–10` for final evaluation
  
#### Transformer parameters

- `encoder_type = "transformer"` or `"rnn"`
- `discriminator_type = "transformer"` or `"rnn"`
- `d_model = 24` — transformer embedding size
- `nhead = 4` — must divide `d_model`
- `dim_feedforward = 96` — usually `4 * d_model`
- `dropout = 0.1`
- `max_seq_len = 24` — should match `seq_len`

### Notes: 
For debugging and testing purposes:

In `discriminative_metrics.py`, iterations = 100. When actually running, use iterations = 10000

In `predictive_metrics.py`, iterations = 100. When actually running, use iterations = 5000


In [3]:
# TimeGAN Model Selection + Parameters 
sys.argv = [
    "notebook",
    "--data_name", "stock",
    "--seq_len", "24",
    "--z_dim", "6",
    "--hidden_dim", "24",
    "--num_layer", "2",
    "--iteration", "10",
    "--batch_size", "128",
    "--metric_iteration", "1",
    "--device", "cpu",
    "--encoder_type", "transformer",
    "--discriminator_type", "rnn",
    "--d_model", "24",
    "--nhead", "4",
    "--dim_feedforward", "96",
    "--dropout", "0.1",
    "--max_seq_len", "24",
    "--name", "notebook_transformer_test"
]

opt_stock = Options().parse()

In [4]:
no = 1000
seq_len = opt.seq_len
dim = opt.z_dim

ori_data = np.random.rand(no, seq_len, dim)

print(ori_data.shape)

(1000, 24, 6)


In [11]:
# Data Generation + Loading

# Synthetic Sine Data
'''
no = amount of synthetic time-series samples
seq_len = the number of time steps in each sample
dim = 5 sine-wave features
'''
sine_data = sine_data_generation(no=10000, seq_len=24, dim=5)

# Real Stock Data 
stock_data = real_data_loading("stock", 24) # Load 24 sequences (w/ 6 features)

In [12]:
%%capture
# Call and Train the Model on Stocks

model_stock = TimeGAN(opt, stock_data)
model_stock.train()

In [21]:
# %%capture
# # Call and Train the Model on Sine data

# model_sine = TimeGAN(opt, sine_data)
# model_sine.train()

In [17]:
generated_data = model_stock.generated_data

print("Generated samples:", len(generated_data))
print("First generated sample shape:", generated_data[0].shape)

Generated samples: 128
First generated sample shape: (24, 6)


In [18]:
%%capture
# Evaluation of Generated Data
from lib.metrics.discriminative_metrics import discriminative_score_metrics

disc_score = discriminative_score_metrics(ori_data, generated_data)

In [19]:
print("Discriminative score:", disc_score)

Discriminative score: 0.29646017699115046


In [25]:
metric_iteration = 10

disc_scores = []
pred_scores = []

for _ in range(metric_iteration):
    disc_scores.append(discriminative_score_metrics(ori_data, generated_data))
    # pred_scores.append(predictive_sscore_metrics(ori_data, generated_data))

print("Discriminative score:", np.mean(disc_scores), "+/-", np.std(disc_scores))
# print("Predictive score:", np.mean(pred_scores), "+/-", np.std(pred_scores))

Discriminative score: 0.28407079646017697 +/- 0.10344523324745103
